## SETUP


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import random
from tqdm import tqdm
import torchvision.transforms as transforms
from transformers import ViTModel

# =====================================================================
# 1. TỰ ĐỘNG QUÉT TOÀN BỘ ẢNH (CHỐNG SAI ĐƯỜNG DẪN / SAI TÊN FOLDER)
# =====================================================================
print("🔍 Đang tự động tìm ảnh trong /kaggle/input...")

all_image_paths = []
# Quét vô điều kiện tất cả file ảnh có trong toàn bộ thư mục input
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            all_image_paths.append(os.path.join(root, file))

if len(all_image_paths) == 0:
    raise RuntimeError("❌ LỖI: Không tìm thấy bất kỳ file ảnh nào! Hãy chắc chắn bạn đã add Dataset ở cột bên phải.")

# Đặt gốc tọa độ là /kaggle/input để đồng bộ đường dẫn tuyệt đối
DATA_ROOT = "/kaggle/input"
metadata_path = "metadata.csv"
save_dir = "checkpoints"

# =====================================================================
# 2. TỰ ĐỘNG TRÍCH XUẤT ID HÃNG VÀ TẠO BẢNG METADATA CHUẨN
# =====================================================================
records = []
for full_path in all_image_paths:
    # filename là đường dẫn tương đối tính từ /kaggle/input trở đi
    rel_path = os.path.relpath(full_path, DATA_ROOT)
    # logo_id tự động lấy tên thư mục chứa trực tiếp file ảnh đó làm tên hãng
    logo_id = os.path.basename(os.path.dirname(full_path))
    records.append({'filename': rel_path, 'logo_id': logo_id})

df = pd.DataFrame(records)
df.to_csv(metadata_path, index=False)

print(f"\n✅ QUÉT DỮ LIỆU THÀNH CÔNG!")
print(f"📸 Tìm thấy tổng cộng: {len(df)} ảnh.")
print(f"🏷️ Xác định được: {df['logo_id'].nunique()} hãng logo khác nhau.")
print(f"📝 Đã ghi nhận file cấu trúc: {metadata_path}")

# Xem trước kiểm tra dữ liệu thực tế
print("\n👀 Kiểm tra 5 dòng đầu tiên trong Metadata:")
print(df.head())

# =====================================================================
# 3. ĐỊNH NGHĨA DATASET TRIPLET (Tích hợp sẵn Augmentation)
# =====================================================================
class LogoTripletDataset(Dataset):
    def __init__(self, metadata, root_dir, mode='train'):
        self.metadata = metadata
        self.root_dir = root_dir
        self.mode = mode
        self.transform = self._default_transform(mode)
        self.triplets = self._build_triplets()

    def _default_transform(self, mode):
        if mode == 'train':
            return transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomRotation(15),
                transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
                transforms.ColorJitter(brightness=0.3, contrast=0.3),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            return transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def _build_triplets(self):
        logo_to_files = self.metadata.groupby('logo_id')['filename'].apply(list).to_dict()
        triplets = []
        all_logos = list(logo_to_files.keys())
        for logo, files in logo_to_files.items():
            n = len(files)
            if n >= 2:
                for i in range(min(20, n)):
                    for j in range(i+1, min(20, n)):
                        anchor, positive = files[i], files[j]
                        neg_logo = random.choice(all_logos)
                        while neg_logo == logo: 
                            neg_logo = random.choice(all_logos)
                        negative = random.choice(logo_to_files[neg_logo])
                        triplets.append((anchor, positive, negative))
        random.shuffle(triplets)
        return triplets

    def __len__(self): 
        return len(self.triplets)

    def __getitem__(self, idx):
        a, p, n = self.triplets[idx]
        img_a = Image.open(os.path.join(self.root_dir, a)).convert('RGB')
        img_p = Image.open(os.path.join(self.root_dir, p)).convert('RGB')
        img_n = Image.open(os.path.join(self.root_dir, n)).convert('RGB')
        return self.transform(img_a), self.transform(img_p), self.transform(img_n)

# =====================================================================
# 4. ĐỊNH NGHĨA KIẾN TRÚC MÔ HÌNH TRIPLET
# =====================================================================
class ViTEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224')
    def forward(self, x):
        return self.vit(x).last_hidden_state[:, 0, :]

class TripletNetwork(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
    def forward(self, a, p=None, n=None):
        if p is None: 
            return self.backbone(a) # Sử dụng cho khâu Inference / HNSW / Test ảnh
        return self.backbone(a), self.backbone(p), self.backbone(n)

## TRAIN

In [ ]:
EPOCHS = 30
LEARNING_RATE = 1e-4
MARGIN = 0.5 
BATCH_SIZE = 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. CHIA DỮ LIỆU THEO LOGO ID (CHỐNG DATA LEAKAGE)
unique_logos = df['logo_id'].unique()
np.random.seed(42)
np.random.shuffle(unique_logos)
split = int(0.8 * len(unique_logos))
train_logos, val_logos = unique_logos[:split], unique_logos[split:]

df_train = df[df['logo_id'].isin(train_logos)].reset_index(drop=True)
df_val = df[df['logo_id'].isin(val_logos)].reset_index(drop=True)

train_dataset = LogoTripletDataset(df_train, DATA_ROOT, mode='train')
val_dataset = LogoTripletDataset(df_val, DATA_ROOT, mode='val')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"🚀 Triplet Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# 2. KHỞI TẠO MODEL & OPTIMIZER
backbone = ViTEncoder()
for param in backbone.parameters(): param.requires_grad = False
for param in backbone.vit.encoder.layer[10:].parameters(): param.requires_grad = True # Mở khóa block 10+
for param in backbone.vit.pooler.parameters(): param.requires_grad = True

model = TripletNetwork(backbone).to(DEVICE)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=0.05)
criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
scaler = torch.amp.GradScaler('cuda')

# 3. VÒNG LẶP TRAIN
best_val_loss = float('inf')
patience, patience_counter = 5, 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    for a, p, n in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        a, p, n = a.to(DEVICE), p.to(DEVICE), n.to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            ea, ep, en = model(a, p, n)
            ea, ep, en = F.normalize(ea), F.normalize(ep), F.normalize(en)
            loss = criterion(ea, ep, en)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        t_loss += loss.item()
        with torch.no_grad():
            t_correct += (F.pairwise_distance(ea, ep) < F.pairwise_distance(ea, en)).sum().item()
            t_total += a.size(0)
            
    # VALIDATION
    model.eval()
    v_loss, v_correct, v_total = 0, 0, 0
    with torch.no_grad():
        for a, p, n in val_loader:
            a, p, n = a.to(DEVICE), p.to(DEVICE), n.to(DEVICE)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                ea, ep, en = model(a, p, n)
                ea, ep, en = F.normalize(ea), F.normalize(ep), F.normalize(en)
                loss = criterion(ea, ep, en)
            v_loss += loss.item()
            v_correct += (F.pairwise_distance(ea, ep) < F.pairwise_distance(ea, en)).sum().item()
            v_total += a.size(0)

    train_loss, train_acc = t_loss/len(train_loader), t_correct/t_total
    val_loss, val_acc = v_loss/len(val_loader), v_correct/v_total
    scheduler.step(val_loss)
    
    for k, v in zip(history.keys(), [train_loss, val_loss, train_acc, val_acc]): history[k].append(v)
    print(f"Epoch {epoch+1} - Loss: {train_loss:.4f}/{val_loss:.4f} | Acc: {train_acc:.4f}/{val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        os.makedirs(save_dir, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(save_dir, 'best_model.pth'))
    else:
        patience_counter += 1
        if patience_counter >= patience: break

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix

# ==========================================
# 1. VẼ BIỂU ĐỒ LOSS & ACCURACY TỪ HISTORY
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(range(1, len(history['train_loss']) + 1), history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(range(1, len(history['val_loss']) + 1), history['val_loss'], label='Val Loss', marker='o')
axes[0].set_title('Training and Validation Loss')
axes[0].set_xlabel('Epochs')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(range(1, len(history['train_acc']) + 1), history['train_acc'], label='Train Acc', marker='s', color='green')
axes[1].plot(range(1, len(history['val_acc']) + 1), history['val_acc'], label='Val Acc', marker='s', color='red')
axes[1].set_title('Training and Validation Accuracy')
axes[1].set_xlabel('Epochs')
axes[1].legend()
axes[1].grid(True)
plt.show()

# ==========================================
# 2. TÌM THRESHOLD TỐI ƯU & VẼ MA TRẬN NHẦM LẪN
# ==========================================
print("Đang tìm Threshold tối ưu trên tập Validation...")
model.load_state_dict(torch.load(os.path.join(save_dir, 'best_model.pth'), map_location=DEVICE))
model.eval()

all_labels = []
all_distances = []

with torch.no_grad():
    for a, p, n in val_loader:
        a, p, n = a.to(DEVICE), p.to(DEVICE), n.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            ea, ep, en = model(a, p, n)
            
        ea = F.normalize(ea, p=2, dim=1)
        ep = F.normalize(ep, p=2, dim=1)
        en = F.normalize(en, p=2, dim=1)
        
        dist_p = F.pairwise_distance(ea, ep, p=2).cpu().numpy()
        dist_n = F.pairwise_distance(ea, en, p=2).cpu().numpy()
        
        all_distances.extend(dist_p)
        all_labels.extend([1] * len(dist_p))
        
        all_distances.extend(dist_n)
        all_labels.extend([0] * len(dist_n))

all_labels = np.array(all_labels)
all_distances = np.array(all_distances)

# Thuật toán tìm Threshold tốt nhất
best_threshold = 0.0
best_acc = 0.0

for t in np.arange(0.1, 2.0, 0.05):
    preds = (all_distances < t).astype(int)
    acc = (preds == all_labels).mean()
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"🎯 Đã tìm thấy Ngưỡng (Threshold) tối ưu: {best_threshold:.2f}")
print(f"📊 Độ chính xác phân loại cặp ảnh tối đa: {best_acc*100:.2f}%")

# Vẽ lại Ma trận với Threshold xịn nhất
final_preds = (all_distances < best_threshold).astype(int)
cm = confusion_matrix(all_labels, final_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Khác nhau (0)', 'Giống nhau (1)'],
            yticklabels=['Khác nhau (0)', 'Giống nhau (1)'])
plt.title(f'Confusion Matrix (Optimal Threshold = {best_threshold:.2f})')
plt.ylabel('Thực tế')
plt.xlabel('Mô hình dự đoán')
plt.show()


## HNSW

In [ ]:
import faiss
import numpy as np
import pickle
import os

# 1. Dataset trích xuất vector mới (Lấy tên thư mục cha làm ID Logo để đồng bộ)
class InferenceDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert('RGB')
        img_tensor = self.transform(img)
        img_id = os.path.basename(os.path.dirname(path)) # Lấy tên thư mục cha
        return img_tensor, img_id

def build_faiss_index_hnsw_cosine(model, dataloader, device):
    model.eval()
    embeddings = []
    ids = []
    with torch.no_grad():
        for img_tensors, img_ids in tqdm(dataloader, desc="Encoding images"):
            img_tensors = img_tensors.to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                # Gọi thẳng model(img) - p và n tự động là None theo cấu trúc TripletNetwork
                batch_embs = model(img_tensors) 
            embeddings.append(batch_embs.cpu().numpy().astype('float32'))
            ids.extend(img_ids)

    embeddings = np.vstack(embeddings)
    faiss.normalize_L2(embeddings) # Chuẩn hóa để dùng Metric Inner Product làm Cosine
    
    d = embeddings.shape[1]
    M = 32
    print(f"Đang khởi tạo HNSW Cosine Index (d={d}, M={M})...")
    index = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 200
    index.add(embeddings)
    return index, ids

# 2. Khởi chạy tạo Index (Sử dụng bộ val_transform chuẩn không augment từ Dataset)
eval_transform = val_dataset.transform 

model.load_state_dict(torch.load(os.path.join(save_dir, 'best_model.pth'), map_location=DEVICE))
model.to(DEVICE)

inference_dataset = InferenceDataset(all_image_paths, eval_transform)
inference_loader = DataLoader(inference_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

index, ids = build_faiss_index_hnsw_cosine(model, inference_loader, DEVICE)
index.hnsw.efSearch = 128

# 3. Lưu index và pkl
save_output_dir = '/kaggle/working/SAVE'
os.makedirs(save_output_dir, exist_ok=True)
index_path = os.path.join(save_output_dir, 'hnsw_index.bin')
ids_path = os.path.join(save_output_dir, 'hnsw_ids.pkl')

faiss.write_index(index, index_path)
with open(ids_path, 'wb') as f:
    pickle.dump(ids, f)
print(f"✅ Đã lưu FAISS Index và danh sách IDs thành công!")

In [ ]:
import faiss
import pickle
import os

# Đảm bảo đường dẫn lưu trữ
save_dir = '/kaggle/working/SAVE'
os.makedirs(save_dir, exist_ok=True)

index_path = os.path.join(save_dir, 'hnsw_index.bin')
ids_path = os.path.join(save_dir, 'hnsw_ids.pkl')

# 1. Lưu cấu trúc FAISS index
faiss.write_index(index, index_path)

# 2. Lưu danh sách IDs (tên file) đi kèm
with open(ids_path, 'wb') as f:
    pickle.dump(ids, f)

print(f"✅ Đã lưu FAISS index thành công tại: {index_path}")
print(f"✅ Đã lưu danh sách IDs thành công tại: {ids_path}")

## SIMPLE TEST

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import faiss

# Bốc ngẫu nhiên 1 ảnh từ danh sách gốc để làm Query
query_path = random.choice(all_image_paths)
print(f"🎯 Ảnh Query gốc: {query_path}")

def get_query_embedding(image_path, model, device, transform):
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            emb = model(img_tensor).cpu().numpy().astype('float32').reshape(1, -1)
    faiss.normalize_L2(emb)
    return emb

# Trích xuất đặc trưng và Tìm kiếm Top 10 trong FAISS
eval_transform = val_dataset.transform
query_emb = get_query_embedding(query_path, model, DEVICE, eval_transform)
scores, indices = index.search(query_emb, 10)

# Khởi tạo khung vẽ: 1 ảnh Query + 5 ảnh kết quả
fig, axes = plt.subplots(1, 6, figsize=(20, 4))

# 1. Vẽ ảnh Query bên trái cùng
try:
    query_img_pil = Image.open(query_path).convert('RGB')
    query_label = os.path.basename(os.path.dirname(query_path))
    axes[0].imshow(query_img_pil)
    axes[0].set_title(f"QUERY\n[{query_label}]", color='blue', fontweight='bold')
except Exception as e:
    axes[0].text(0.5, 0.5, f"Lỗi đọc ảnh\nQuery", ha='center', va='center')
axes[0].axis('off')

# 2. Vòng lặp đổ dữ liệu vào 5 ô tiếp theo
plot_idx = 1
for i, idx in enumerate(indices[0]):
    if plot_idx > 5: 
        break # Đủ 5 ảnh tương đồng thì dừng
        
    match_path = all_image_paths[idx]
    
    # Nếu trùng chính xác với ảnh đang đem đi test thì bỏ qua để lấy ảnh khác cùng hãng
    if match_path == query_path: 
        continue 
        
    try:
        # Đọc ảnh từ Database dựa vào index chuẩn của FAISS
        match_img_pil = Image.open(match_path).convert('RGB')
        match_label = ids[idx] # Lấy tên hãng
        score = float(scores[0][i])
        pct = max(0.0, score * 100.0)
        
        # Đổ ảnh vào ô tương ứng
        axes[plot_idx].imshow(match_img_pil)
        axes[plot_idx].set_title(f"{match_label}\n{pct:.1f}% giống")
    except Exception as e:
        # Nếu lỗi không đọc được ảnh, in chữ thông báo thay vì để ô trắng rỗng
        axes[plot_idx].text(0.5, 0.5, f"Lỗi ảnh\nIndex {idx}", ha='center', va='center', color='red')
        print(f"❌ Không thể hiển thị ảnh tại index {idx}. Đường dẫn: {match_path}. Lỗi: {e}")
        
    axes[plot_idx].axis('off')
    plot_idx += 1

plt.tight_layout()
plt.show()

## TEST LOGO 


In [ ]:
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from PIL import Image
import torch
import faiss

# URL ảnh Logo test từ Internet
image_url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT3TdwtDJjIhEkM6pfyvjwaqWo0Lvh0ZqJD9g&s"
# image_url = "https://upload.wikimedia.org/wikipedia/sco/thumb/d/d3/Starbucks_Corporation_Logo_2011.svg/1280px-Starbucks_Corporation_Logo_2011.svg.png"
print(f"Đang tải ảnh từ: {image_url}")

try:
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(image_url, headers=headers, timeout=5)
    response.raise_for_status()
    raw_img = Image.open(BytesIO(response.content))
    
    if raw_img.mode in ('RGBA', 'LA') or (raw_img.mode == 'P' and 'transparency' in raw_img.info):
        raw_img = raw_img.convert('RGBA')
        white_bg = Image.new('RGBA', raw_img.size, (255, 255, 255, 255))
        white_bg.paste(raw_img, (0, 0), raw_img)
        query_img = white_bg.convert('RGB')
    else:
        query_img = raw_img.convert('RGB')
    print("✅ Tải và xử lý ảnh thành công!\n")
except Exception as e:
    print(f"❌ Lỗi khi tải ảnh: {e}")
    query_img = None

if query_img is not None:
    eval_transform = val_dataset.transform
    img_tensor = eval_transform(query_img).unsqueeze(0).to(DEVICE)
    
    model.eval()
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            emb = model(img_tensor).cpu().numpy().astype('float32').reshape(1, -1)
    faiss.normalize_L2(emb)
    
    # Tìm Top 5
    top_k = 5
    scores, indices = index.search(emb, top_k)
    
    # ==========================================
    # HIỂN THỊ TRỰC QUAN BẰNG HÌNH ẢNH
    # ==========================================
    fig, axes = plt.subplots(1, top_k + 1, figsize=(20, 4))
    
    # Vẽ ảnh Query từ Internet
    axes[0].imshow(query_img)
    axes[0].set_title("QUERY\n(Ảnh Internet)", color='blue', fontweight='bold')
    axes[0].axis('off')
    
    # Vẽ Top 5 kết quả bốc từ Database lên
    for i, idx in enumerate(indices[0]):
        score = float(scores[0][i])
        pct = max(0.0, score * 100.0)
        match_path = all_image_paths[idx] # Lấy đường dẫn ảnh gốc trong máy
        match_label = ids[idx]
        
        match_img_pil = Image.open(match_path).convert('RGB')
        
        axes[i+1].imshow(match_img_pil)
        axes[i+1].set_title(f"{match_label}\n{pct:.1f}% giống")
        axes[i+1].axis('off')

    plt.tight_layout()
    plt.show()